# 06 · Contraction with einsum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/06-contraction-with-einsum.ipynb)

*Part IV · exercise · 15 min*

> 🇪🇸 **Contracción con einsum** — Una sola notación para el producto punto, el producto matricial y un lote de imágenes.

One notation for the dot product, the matrix product, and a batch of images.

## What you will be able to do

- State the einsum rule: an index missing after the arrow is summed over.
- Contract the colour axis of one image, and of a whole batch, with one call each.
- Write trace, transpose and the matrix product as `einsum` and check them against NumPy.
- Build a full similarity matrix between 1797 images with a single contraction.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from skimage import data

photo = data.immunohistochemistry().astype(float)         # (512, 512, 3)
batch = np.stack([photo, data.astronaut().astype(float)])  # (2, 512, 512, 3)
w = np.array([0.2125, 0.7154, 0.0721])                     # RGB -> grayscale weights
A = np.array([[1., 2.], [3., 4.]])
B = np.array([[5., 6.], [7., 8.]])
print(photo.shape, batch.shape)

## Why this matters

> 🇪🇸 Los sistemas de recomendación y de búsqueda ordenan los resultados con el
> producto punto entre el vector de un usuario y millones de vectores de
> artículos. Esa contracción *es* la señal de ranking.

Recommendation and search systems rank items by the dot product between a user
vector and every item vector — one user against millions of items, many times
per second. That contraction *is* the ranking signal. Sum over the wrong axis
and every user gets wrong results.

### The rule, again

An index that appears in the inputs but **not** after the arrow is **summed
over**. An index that appears after the arrow is **kept**.

That is the whole of `einsum`. Everything below is that one sentence applied.

## Exercise 1 — contract the colour axis

> 🇪🇸 Contrae el eje de color.

In [ ]:
# TODO 1: With einsum, convert `photo` to grayscale by contracting the colour
#         axis against w. Result shape (512, 512).

# TODO 2: Do the same for the whole batch in ONE einsum call -> (2, 512, 512).

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
gray       = np.einsum('hwc,c->hw',   photo, w)     # (512, 512)
gray_batch = np.einsum('nhwc,c->nhw', batch, w)     # (2, 512, 512)
print(gray.shape, gray_batch.shape)

# `c` appears in the inputs but not after the arrow, so it is SUMMED OVER —
# that is the contraction. `n`, `h`, `w` appear after the arrow, so they are
# KEPT. Adding a batch axis costs exactly one letter.

## Exercise 2 — Chapter 2, rewritten as contractions

> 🇪🇸 Las operaciones del capítulo 2, escritas como contracciones.

In [ ]:
# TODO 3: Write these Chapter 2 operations as einsum and check each against
#         NumPy:
#           (a) trace          (eq 2.48)
#           (b) transpose      (eq 2.3)
#           (c) matrix product (eq 2.5)

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
print(np.einsum('ii->', A),        np.trace(A))       # trace
print(np.einsum('ij->ji', A),     A.T, sep="\n")       # transpose
print(np.einsum('ik,kj->ij', A, B), A @ B, sep="\n")   # matrix product

for got, want in [(np.einsum('ii->', A), np.trace(A)),
                  (np.einsum('ij->ji', A), A.T),
                  (np.einsum('ik,kj->ij', A, B), A @ B)]:
    assert np.allclose(got, want)
print("all three agree")

# Trace: the repeated `i` with nothing after the arrow sums the diagonal.
# Transpose: no index is summed at all — einsum is just relabelling axes.
# Matrix product: `k` is shared and dropped, so it is the contracted axis.

## Exercise 3 — every pair of 1797 images, in one call

> 🇪🇸 Todos los pares de 1797 imágenes, en una sola llamada.

This one matters beyond the exercise: it is the same operation a search engine
runs, and it is the bridge to the distance and similarity questions in the
Kahoot below.

In [ ]:
# TODO 4: Flatten the digits to (1797, 64) and compute the (1797, 1797)
#         similarity matrix between every pair of digit images with one einsum.
#
#         Then, for the quiz: normalize each row to unit length first and do it
#         again. That second version is COSINE SIMILARITY — the dot product
#         divided by the two norms. The unnormalized one is dominated by how
#         much ink each digit has, not by its shape.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
D = load_digits().images.reshape(1797, -1)          # (1797, 64)

S = np.einsum('id,jd->ij', D, D)                    # (1797, 1797)
print(S.shape, S.size)                              # 3,229,209 pairwise scores

# Cosine similarity: the same contraction on unit-length rows.
norms = np.linalg.norm(D, axis=1, keepdims=True)
Dn = D / np.where(norms == 0, 1.0, norms)
C = np.einsum('id,jd->ij', Dn, Dn)
print(C.diagonal()[:3])                             # ~1.0 — each digit matches itself

# EUCLIDEAN DISTANCE is the square root of summed squared differences, and it is
# built from the same contraction:
sq = (D ** 2).sum(1)
dist = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * S, 0))
print(np.round(dist[0, :4], 1))

## What just happened

`c` appears in the inputs but not after the arrow, so it is **summed over** —
that is the contraction. `n`, `h`, `w` appear after the arrow, so they are
**kept**. Adding a batch axis costs exactly one letter.

This is why `einsum` is worth learning: **the same expression works for one image
or for a million**, and it reads like the mathematics in Chapter 2.

> 🇪🇸 La misma expresión sirve para una imagen o para un millón, y se lee como
> las matemáticas del capítulo 2.

Keep it in mind for section 10, where a single `einsum` string contracts three
axes at once: `'ijk,ia,jb,kc->abc'`. That is why einsum came first.

---

## Done with this section

Next up: **07 · Inverses and the pseudoinverse** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/07-inverses-and-pseudoinverse.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)